# LLaMA QLoRA Fine-Tuning for Connections

Fine-tunes LLaMA 3.1 with **4-bit QLoRA** on the Connections training split, then evaluates
on the test split using the same metrics as the other notebooks.

4-bit quantization reduces VRAM from ~16 GB (fp16) to ~6–8 GB, making it compatible with
consumer GPUs such as an RTX 5070 (12 GB).

**Prerequisites:**
- `pip install transformers peft torch datasets huggingface_hub bitsandbytes`
- Set the `HF_API_KEY` environment variable with your Hugging Face token
  (needs access to `meta-llama/Llama-3.1-8B`).

In [1]:
import os
from huggingface_hub import login
from dotenv import load_dotenv
import torch

load_dotenv()
login(token=os.environ["HF_API_KEY"])

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")

c:\Users\bioin\miniconda3\envs\connections\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from data_loader import get_cv_folds, get_train_test_split

N_FOLDS = 5

folds = get_cv_folds(n_folds=N_FOLDS)
ds_train, ds_test = get_train_test_split()

print(f"CV folds ({N_FOLDS}):")
for f in folds:
    print(f"  Fold {f.fold}: Train={len(f.train)}  Val={len(f.val)}")
print(f"Full train split: {len(ds_train)}  |  Test split: {len(ds_test)}")

CV folds (5):
  Fold 0: Train=521  Val=131
  Fold 1: Train=521  Val=131
  Fold 2: Train=522  Val=130
  Fold 3: Train=522  Val=130
  Fold 4: Train=522  Val=130
Full train split: 521  |  Test split: 131


## 1. Cross-Validation: Fine-tune & Evaluate

### NOTE: Skip 1 and 2 for actual model generation. These are for eval purposes
Trains one LoRA adapter per fold, evaluates on the held-out validation split, and stores per-fold metrics.
Each fold's adapter is saved to `adapters/llama-lora-cv{fold}` so runs can be resumed.

In [3]:
import torch
from conn.llama_fine_tuning import finetune_llama_lora
from conn.solvers.llama import LlamaSolver
from conn import accuracy_min_swaps, accuracy_zero_one, evaluate

cv_results = []

for fold in folds:
    print(f"\n{'='*50}")
    print(f"Fold {fold.fold + 1}/{N_FOLDS}  (train={len(fold.train)}  val={len(fold.val)})")
    print('='*50)

    model, tokenizer, stats = finetune_llama_lora(
        puzzles=list(fold.train),
        epochs=3,
        batch_size=2,
        learning_rate=2e-4,
        lora_r=8,
        lora_alpha=16,
        use_4bit=True,
        adapter_output_dir=f"./adapters/llama-lora-cv{fold.fold}",
        verbose=True,
    )

    solver = LlamaSolver(model=model, tokenizer=tokenizer, k=0, max_retries=4)
    results = evaluate(
        fold.val,
        metric_fns={"zero_one": accuracy_zero_one, "min_swaps": accuracy_min_swaps},
        solver_fn=solver.solve,
        verbose=False,
    )

    acc, n_eval = results["zero_one"]
    mean_swaps, _ = results["min_swaps"]
    cv_results.append({
        "fold": fold.fold,
        "zero_one": acc,
        "min_swaps": mean_swaps,
        "n_eval": n_eval,
        "epoch_losses": stats.average_losses,
    })
    print(f"\nFold {fold.fold} | zero_one={acc:.4f}  min_swaps={mean_swaps:.2f}  (n={n_eval})")
    print(f"Epoch losses: {[f'{l:.4f}' for l in stats.average_losses]}")

    del model, solver
    torch.cuda.empty_cache()


Fold 1/5  (train=521  val=131)
Starting LLaMA 4-bit QLoRA SFT on cuda for 3 epochs...


Epoch 1/3:  11%|█▏        | 30/261 [01:37<12:30,  3.25s/it, loss=0.4069]


KeyboardInterrupt: 

## 2. CV Results Summary

In [ ]:
import numpy as np

print("=== Cross-Validation Results ===\n")
for r in cv_results:
    losses = [f"{l:.4f}" for l in r["epoch_losses"]]
    print(f"  Fold {r['fold']} | zero_one={r['zero_one']:.4f}  min_swaps={r['min_swaps']:.2f}  n={r['n_eval']}  losses={losses}")

accs = [r["zero_one"] for r in cv_results]
swaps = [r["min_swaps"] for r in cv_results]
print(f"\nMean zero-one : {np.mean(accs):.4f} ± {np.std(accs):.4f}")
print(f"Mean min swaps: {np.mean(swaps):.2f} ± {np.std(swaps):.2f}")

## 3. Final Model: Fine-tune on All Training Data

Trains a single adapter on the full training split — no held-out fold.
This is the model used for final evaluation and deployment.

In [3]:
from conn.llama_fine_tuning import finetune_llama_lora

model, tokenizer, stats = finetune_llama_lora(
    puzzles=list(ds_train),
    epochs=3,
    batch_size=2,
    learning_rate=2e-4,
    lora_r=8,
    lora_alpha=16,
    use_4bit=True,
    adapter_output_dir="./adapters/llama-lora-final",
    verbose=True,
)

print(f"\nFinal model training complete — {stats.steps} steps")
print("Epoch losses:", [f"{l:.4f}" for l in stats.average_losses])

Starting LLaMA 4-bit QLoRA SFT on cuda for 3 epochs...


Epoch 1/3: 100%|██████████| 261/261 [10:59<00:00,  2.53s/it, loss=0.0771]


Epoch 1/3 | Average Loss: 0.3406


Epoch 2/3: 100%|██████████| 261/261 [11:10<00:00,  2.57s/it, loss=0.0620]


Epoch 2/3 | Average Loss: 0.2376


Epoch 3/3: 100%|██████████| 261/261 [11:08<00:00,  2.56s/it, loss=0.0396]


Epoch 3/3 | Average Loss: 0.1533
Adapter saved to adapters\llama-lora-final
Training stats saved to adapters\llama-lora-final\training_stats.json

Final model training complete — 783 steps
Epoch losses: ['0.3406', '0.2376', '0.1533']


## 3a. (Optional) Load base model (no fine-tuning)

Run this **instead of** 3b if you want to evaluate the **base** LLaMA without any fine-tuned adapter. Same `model`, `tokenizer` interface for section 4.

In [5]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from conn.llama_fine_tuning import DEFAULT_MODEL_NAME

use_4bit = True
if use_4bit and device.type == "cuda":
    from transformers import BitsAndBytesConfig
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )
    model = AutoModelForCausalLM.from_pretrained(
        DEFAULT_MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto",
    )
else:
    model = AutoModelForCausalLM.from_pretrained(
        DEFAULT_MODEL_NAME,
        torch_dtype=torch.float16 if device.type == "cuda" else torch.float32,
    )
    model.to(device)

tokenizer = AutoTokenizer.from_pretrained(DEFAULT_MODEL_NAME, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model.eval()
print("Loaded base model (no adapter).")

Loading weights: 100%|██████████| 291/291 [00:21<00:00, 13.74it/s, Materializing param=model.norm.weight]                              


Loaded base model (no adapter).


## 3b. (Optional) Load a Previously Saved Final Adapter

Skip this cell if you just trained above.

In [4]:
from conn.llama_fine_tuning import load_llama_lora

model, tokenizer = load_llama_lora("./adapters/llama-lora-final")

Loading weights: 100%|██████████| 291/291 [00:13<00:00, 21.93it/s, Materializing param=model.norm.weight]                               


## 4. Evaluate Final Model (zero-shot)

In [4]:
from conn.solvers.llama import LlamaSolver
from conn import accuracy_min_swaps, accuracy_zero_one, evaluate

print(f"Using device {device}")

solver_ft_zero = LlamaSolver(
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=80,
    k=0,
    max_retries = 4,
    device=device
)

results_zero = evaluate(
    ds_test,
    metric_fns={"zero_one": accuracy_zero_one, "min_swaps": accuracy_min_swaps},
    solver_fn=solver_ft_zero.solve,
    verbose=True,
)

acc, n_eval = results_zero["zero_one"]
mean_swaps, _ = results_zero["min_swaps"]

print(f"\nFinal model k=0 | Zero-one accuracy: {acc:.4f}  (n={n_eval})")
print(f"Final model k=0 | Mean min swaps: {mean_swaps:.2f}")

Using device cuda
LlamaSolver: Using device cuda
Invalid LLM output, retrying (1/4)
Invalid LLM output, retrying (2/4)
Invalid LLM output, retrying (3/4)
Invalid LLM output, retrying (4/4)
[23:24:41 → 23:25:38 | 56.1s generate=56.1s] Puzzle 1 solved, min_swaps=12.0
[23:25:38] WARNING: Invalid or hallucinated output for puzzle 1: [['CRUISE', 'ECHO', 'MIMIC', 'PARROT'], ['COPY', 'Imitate', 'Replicate', 'Reproduce'], ['FURIOUS', 'Irate', 'Livid', 'Wrathful'], ['Mild', 'Mellow', 'Petty', 'Tame']]
EXPECTED: [['DENMARK', 'GREECE', 'POLAND', 'PORTUGAL'], ['COPY', 'ECHO', 'MIMIC', 'PARROT'], ['CRUISE', 'HOLLAND', 'PETTY', 'WAITS'], ['DILL', 'LIVID', 'MILD', 'MIX']]
--- RAW LLM OUTPUT ---
GROUP 1: CRUISE, ECHO, MIMIC, PARROT
GROUP 2: COPY, Imitate, Replicate, Reproduce
GROUP 3: FURIOUS, Irate, Livid, Wrathful
GROUP 4: Mild, Mellow, Petty, Tame
GROUP 1: FURIOUS, Irate, Livid, Wrathful
GROUP 
----------------------
Invalid LLM output, retrying (1/4)
Invalid LLM output, retrying (2/4)
Invalid LLM 

KeyboardInterrupt: 

In [5]:
# Debug single puzzle prediction to inspect hallucination
test_puzzle = list(ds_test)[0]
words16 = test_puzzle.get('words', [])
print('Original 16 words:', words16)

if 'answers' in test_puzzle:
    gold = [ans['words'] for ans in test_puzzle['answers']]
else:
    gold = test_puzzle.get('groups', [])
print('\nExpected Gold Groups:\n', gold)

print('\n--- RAW LLAMA OUTPUT ---')
prompt = solver_ft_zero._build_full_prompt(words16)
raw_response = solver_ft_zero._generate(prompt)
print(raw_response)
print('------------------------\n')

parsed = solver_ft_zero.solve(words16)
print('Parsed Output:\n', parsed)

from conn.metrics import accuracy_zero_one, accuracy_min_swaps
print('\nMetrics:')
print('Zero-one accuracy:', accuracy_zero_one(parsed, gold))
print('Min swaps:', accuracy_min_swaps(parsed, gold))


Original 16 words: ['DENMARK', 'GREECE', 'POLAND', 'PORTUGAL', 'COPY', 'ECHO', 'MIMIC', 'PARROT', 'CRUISE', 'HOLLAND', 'PETTY', 'WAITS', 'DILL', 'LIVID', 'MILD', 'MIX']

Expected Gold Groups:
 [['DENMARK', 'GREECE', 'POLAND', 'PORTUGAL'], ['COPY', 'ECHO', 'MIMIC', 'PARROT'], ['CRUISE', 'HOLLAND', 'PETTY', 'WAITS'], ['DILL', 'LIVID', 'MILD', 'MIX']]

--- RAW LLAMA OUTPUT ---


KeyboardInterrupt: 

## 5. Evaluate Final Model (few-shot, k=10)

In [6]:
K_VAL = 10
solver_ft_few = LlamaSolver(
    model=model,
    tokenizer=tokenizer,
    example_source=ds_train,
    k=K_VAL,
    max_retries=4
)

results_few = evaluate(
    ds_test,
    metric_fns={"zero_one": accuracy_zero_one, "min_swaps": accuracy_min_swaps},
    solver_fn=solver_ft_few.solve,
    verbose=True,
)

acc, n_eval = results_few["zero_one"]
mean_swaps, _ = results_few["min_swaps"]
print(f"\nFinal model k={K_VAL} | Zero-one accuracy: {acc:.4f}  (n={n_eval})")
print(f"Final model k={K_VAL} | Mean min swaps: {mean_swaps:.2f}")

LlamaSolver: Using device cuda
Invalid LLM output, retrying (1/4)
Invalid LLM output, retrying (2/4)


KeyboardInterrupt: 

## 5. Comparison: Base model (no fine-tuning)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

BASE_MODEL = "meta-llama/Llama-3.1-8B-Instruct"

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

base_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)

if device.type == "cuda":
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        quantization_config=bnb_config,
        device_map="auto",
    )
else:
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        torch_dtype=torch.float32,
    ).to(device)

base_model.eval()

solver_base = LlamaSolver(
    model=base_model,
    tokenizer=base_tokenizer,
    example_source=ds_train,
    k=3,
)

results_base = evaluate(
    ds_test,
    metric_fns={"zero_one": accuracy_zero_one, "min_swaps": accuracy_min_swaps},
    solver_fn=solver_base.solve,
    verbose=True,
)

acc, n_eval = results_base["zero_one"]
mean_swaps, _ = results_base["min_swaps"]
print(f"\nBase k=3 | Zero-one accuracy: {acc:.4f}  (n={n_eval})")
print(f"Base k=3 | Mean min swaps: {mean_swaps:.2f}")